# Fabric Workspace GUID Extractor
Extracts all artifact GUIDs from one or more Microsoft Fabric workspaces
and optionally updates the Variable Library definition via the Fabric REST API.
**Usage:**
1. Set `WORKSPACE_NAMES` below to target specific workspaces by name, or leave empty to auto-detect the current workspace.
2. Set `UPDATE_VALUE_SETS = True` and configure `VALUE_SETS_TO_UPDATE` to control which value sets get updated.
   Use `"default"` to update the base variables, or specific value set names (e.g. `"Env-1D"`) for environment overrides.
3. Run all cells.


In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────────
# Add workspace names here to extract GUIDs from multiple workspaces.
# Leave empty to auto-detect the current workspace via notebookutils.
WORKSPACE_NAMES = ["<WorkspaceName>"]

# Set to True to update the Variable Library via the Fabric REST API
UPDATE_VALUE_SETS = True

# Display name of the Variable Library item in Fabric
VARIABLE_LIBRARY_NAME = "<VariableLibraryName>"

# Which workspace contains the Variable Library to update.
# Must be one of the names in WORKSPACE_NAMES above.
VARIABLE_LIBRARY_WORKSPACE = "__current__"

# Map value set names to workspace names. Use "default" to update the base variables.
# Use "__current__" as the workspace name to auto-resolve to the workspace this notebook runs in.
# Only listed targets are touched; unlisted value sets are passed through unchanged.
VALUE_SETS_TO_UPDATE = {
    "default": "__current__",
    "<SetName1>": "<WorkspaceName>-<EnvironmentSuffix>",
    # "<SetName2>": "<WorkspaceName>-<EnvironmentSuffix>",
    # "<SetName3>": "<WorkspaceName>-<EnvironmentSuffix>",
}

In [ ]:
import json
import base64
import time
import requests
from typing import Dict, List, Optional, Tuple

# ── Authenticate via Fabric runtime ──────────────────────────────────────────────
FABRIC_API_BASE = "https://api.fabric.microsoft.com/v1"
TOKEN = notebookutils.credentials.getToken("https://api.fabric.microsoft.com/.default")
HEADERS = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}


def resolve_workspace_names(names: List[str]) -> List[Tuple[str, str]]:
    """Resolve workspace display names to (workspace_id, workspace_name) pairs via the Fabric API."""
    url = f"{FABRIC_API_BASE}/workspaces"
    all_workspaces = []

    # Paginate through all accessible workspaces
    while url:
        response = requests.get(url, headers=HEADERS)
        if response.status_code != 200:
            raise RuntimeError(
                f"Failed to list workspaces: {response.status_code} {response.text}"
            )
        data = response.json()
        all_workspaces.extend(data.get("value", []))
        url = data.get("continuationUri")

    # Build a lookup by displayName
    name_lookup = {ws["displayName"]: ws["id"] for ws in all_workspaces}

    resolved = []
    for name in names:
        ws_id = name_lookup.get(name)
        if ws_id:
            resolved.append((ws_id, name))
        else:
            print(f"WARNING: Workspace '{name}' not found. Available workspaces:")
            for available in sorted(name_lookup.keys()):
                print(f"  - {available}")
            raise RuntimeError(f"Workspace '{name}' not found.")

    return resolved


def poll_lro(response, headers: Dict) -> requests.Response:
    """Poll a Fabric long-running operation until completion, then fetch the result."""
    if response.status_code != 202:
        return response

    location = response.headers.get("Location")

    # Poll operation status until no longer Running
    while True:
        retry_after = int(response.headers.get("Retry-After", 5))
        print(f"  Operation in progress, retrying in {retry_after}s...")
        time.sleep(retry_after)
        response = requests.get(location, headers=headers)

        status = response.json().get("status", "")
        if status not in ("NotStarted", "Running"):
            break

    if response.json().get("status") != "Succeeded":
        return response

    # Fetch the actual result from the /result endpoint
    result_url = f"{location}/result"
    return requests.get(result_url, headers=headers)


# ── Resolve current workspace ────────────────────────────────────────────────────
ctx = notebookutils.runtime.context
CURRENT_WS_ID = ctx.get("workspaceId") or ctx.get("currentWorkspaceId")
CURRENT_WS_NAME = ctx.get("workspaceName") or ctx.get("currentWorkspaceName") or "Unknown"
if not CURRENT_WS_ID:
    raise RuntimeError("Could not detect workspace ID from notebookutils.runtime.context")
print(f"Current workspace: {CURRENT_WS_NAME} ({CURRENT_WS_ID})")

# ── Resolve workspaces ───────────────────────────────────────────────────────────
if WORKSPACE_NAMES:
    WORKSPACES = resolve_workspace_names(WORKSPACE_NAMES)
else:
    WORKSPACES = [(CURRENT_WS_ID, CURRENT_WS_NAME)]

# Ensure current workspace is in the list (needed for "__current__" resolution)
current_in_list = any(ws_id == CURRENT_WS_ID for ws_id, _ in WORKSPACES)
if not current_in_list:
    WORKSPACES.append((CURRENT_WS_ID, CURRENT_WS_NAME))

# Resolve "__current__" in VALUE_SETS_TO_UPDATE and VARIABLE_LIBRARY_WORKSPACE
VALUE_SETS_TO_UPDATE = {
    k: (CURRENT_WS_NAME if v == "__current__" else v)
    for k, v in VALUE_SETS_TO_UPDATE.items()
}
if VARIABLE_LIBRARY_WORKSPACE == "__current__":
    VARIABLE_LIBRARY_WORKSPACE = CURRENT_WS_NAME

print(f"Targeting {len(WORKSPACES)} workspace(s):")
for ws_id, ws_name in WORKSPACES:
    print(f"  {ws_name} -> {ws_id}")

In [ ]:
# Mapping of Fabric artifact display names to variables.json variable names.
# Value can be a string (matches any item type) or a dict {type: var_name} for
# artifacts that appear as multiple types (e.g. Lakehouse + SQLEndpoint).
ARTIFACT_MAPPING = {
    # "<LakehouseDisplayName>": {
    #     "Lakehouse": "LakehouseId",
    #     "SQLEndpoint": "LakehouseSqlEndpointId",
    # },
    # "<DatabaseDisplayName>": "DatabaseId",
    # "<WarehouseDisplayName>": "WarehouseId",
    # "<PipelineDisplayName>": "<VariableName>",
    # "<NotebookDisplayName>": "<VariableName>",
}

# Mapping of ItemReference variable names to their corresponding GUID variable names.
# Used to auto-populate {"itemId": <guid>, "workspaceId": <ws_id>} in value sets.
REFERENCE_MAPPING = {
    # "<LakehouseItem>": "LakehouseId",
    # "<DatabaseItem>": "DatabaseId",
    # "<WarehouseItem>": "WarehouseId",
    # "<PipelineItem>": "<VariableName>",
    # "<NotebookItem>": "<NotebookVariableName>",
}


def get_workspace_items(workspace_id: str) -> List[Dict]:
    """Fetch all items from a workspace via the Fabric REST API."""
    url = f"{FABRIC_API_BASE}/workspaces/{workspace_id}/items"
    response = requests.get(url, headers=HEADERS)

    if response.status_code != 200:
        print(f"ERROR: Failed to fetch items for workspace {workspace_id}")
        print(f"  Status: {response.status_code}  Response: {response.text}")
        return []

    return response.json().get("value", [])


def extract_guids(
    workspace_id: str, workspace_name: str = ""
) -> Tuple[Dict[str, str], Optional[str]]:
    """Extract GUIDs from a single workspace. Returns (guid_mapping, variable_library_id)."""
    items = get_workspace_items(workspace_id)
    guid_mapping = {}
    variable_library_id = None

    label = f"{workspace_name} ({workspace_id})" if workspace_name else workspace_id
    print(f"\n{label} — {len(items)} items")
    print(f"{'Display Name':<45} {'Type':<20} {'ID'}")
    print("-" * 110)

    for item in sorted(items, key=lambda x: x.get("displayName", "")):
        display_name = item.get("displayName", "")
        item_id = item.get("id", "")
        item_type = item.get("type", "")

        print(f"{display_name:<45} {item_type:<20} {item_id}")

        if display_name in ARTIFACT_MAPPING:
            mapping_value = ARTIFACT_MAPPING[display_name]
            if isinstance(mapping_value, dict):
                if item_type in mapping_value:
                    guid_mapping[mapping_value[item_type]] = item_id
            else:
                guid_mapping[mapping_value] = item_id

        if display_name == VARIABLE_LIBRARY_NAME and item_type == "VariableLibrary":
            variable_library_id = item_id

    return guid_mapping, variable_library_id


In [ ]:
# ── Extract GUIDs from all workspaces ────────────────────────────────────────────
all_guid_mappings = {}  # {workspace_id: {var_name: guid}}
variable_library_info = None  # (workspace_id, item_id, workspace_name)

for ws_id, ws_name in WORKSPACES:
    mapping, vl_id = extract_guids(ws_id, ws_name)
    all_guid_mappings[ws_id] = mapping

    # Only use the VL from the explicitly configured workspace
    if vl_id and ws_name == VARIABLE_LIBRARY_WORKSPACE:
        variable_library_info = (ws_id, vl_id, ws_name)

    print(f"\n{'='*110}")
    print(f"GUID MAPPING for {ws_name} ({ws_id}):")
    print(f"{'='*110}")
    for var_name, guid in mapping.items():
        print(f"  {var_name}: {guid}")

if variable_library_info:
    vl_ws_id, vl_item_id, vl_ws_name = variable_library_info
    print(
        f"\nVariable Library '{VARIABLE_LIBRARY_NAME}' found in {vl_ws_name} ({vl_item_id})"
    )
else:
    print(
        f"\nWARNING: Variable Library '{VARIABLE_LIBRARY_NAME}' not found in "
        f"workspace '{VARIABLE_LIBRARY_WORKSPACE}'. Check VARIABLE_LIBRARY_WORKSPACE config."
    )

In [ ]:
# ── Update Variable Library via Fabric REST API (optional) ────────────────────────
def apply_guid_updates(variables: List[Dict], ws_id: str, ws_name: str,
                       guid_mapping: Dict[str, str], label: str) -> List[str]:
    """Update a list of variable entries in place. Returns list of changed variable names.
    Works for both variables.json entries (have 'type'/'note') and value set overrides."""
    updates_made = []
    existing_names = {v["name"] for v in variables}

    print(f"\n  {label}:")
    print(f"    Extracted GUIDs ({len(guid_mapping)}): {list(guid_mapping.keys())}")
    print(f"    Existing variables ({len(existing_names)}): {sorted(existing_names)}")

    for entry in variables:
        var_name = entry.get("name")
        old_value = entry.get("value")
        new_value = None

        if var_name == "WorkspaceId":
            new_value = ws_id
        elif var_name == "WorkspaceName":
            new_value = ws_name
        elif var_name in guid_mapping:
            new_value = guid_mapping[var_name]
        elif var_name in REFERENCE_MAPPING:
            item_guid_var = REFERENCE_MAPPING[var_name]
            if item_guid_var in guid_mapping:
                new_value = {"itemId": guid_mapping[item_guid_var], "workspaceId": ws_id}

        if new_value is not None:
            if old_value != new_value:
                entry["value"] = new_value
                updates_made.append(var_name)
                print(f"    CHANGED {var_name}: {old_value} -> {new_value}")
            else:
                print(f"    MATCH   {var_name}: {old_value}")
        else:
            print(f"    SKIP    {var_name}: not in extracted GUIDs")

    # Add missing workspace identity entries
    for name, value in [("WorkspaceId", ws_id), ("WorkspaceName", ws_name)]:
        if name not in existing_names:
            variables.append({"name": name, "value": value})
            updates_made.append(name)
            print(f"    ADDED   {name}: {value}")

    # Add missing entries for extracted GUIDs not already present
    for var_name, guid in guid_mapping.items():
        if var_name not in existing_names:
            variables.append({"name": var_name, "value": guid})
            updates_made.append(var_name)
            print(f"    ADDED   {var_name}: {guid}")

    for ref_name, guid_var in REFERENCE_MAPPING.items():
        if ref_name not in existing_names and guid_var in guid_mapping:
            ref_value = {"itemId": guid_mapping[guid_var], "workspaceId": ws_id}
            variables.append({"name": ref_name, "value": ref_value})
            updates_made.append(ref_name)
            print(f"    ADDED   {ref_name}: {ref_value}")

    if updates_made:
        print(f"    >> {len(updates_made)} variable(s) updated/added")
    else:
        print("    >> already up to date")

    return updates_made


if UPDATE_VALUE_SETS:
    if not variable_library_info:
        print(
            f"ERROR: Cannot update — Variable Library '{VARIABLE_LIBRARY_NAME}' not found."
        )
    elif not VALUE_SETS_TO_UPDATE:
        print("No targets in VALUE_SETS_TO_UPDATE — nothing to do.")
    else:
        vl_ws_id, vl_item_id, vl_ws_name = variable_library_info
        print(f"Fetching definition for '{VARIABLE_LIBRARY_NAME}' from {vl_ws_name}...")

        # GET current Variable Library definition
        get_url = (
            f"{FABRIC_API_BASE}/workspaces/{vl_ws_id}"
            f"/VariableLibraries/{vl_item_id}/getDefinition"
        )
        response = poll_lro(requests.post(get_url, headers=HEADERS), HEADERS)

        if response.status_code != 200:
            print(
                f"ERROR: Failed to get definition: {response.status_code} {response.text}"
            )
        else:
            definition = response.json()["definition"]

            # Build workspace name -> (ws_id, guid_mapping) lookup
            ws_lookup = {
                ws_name: (ws_id, all_guid_mappings.get(ws_id, {}))
                for ws_id, ws_name in WORKSPACES
            }

            # Validate all targets in VALUE_SETS_TO_UPDATE reference known workspaces
            for target, ws_name in VALUE_SETS_TO_UPDATE.items():
                if ws_name not in ws_lookup:
                    print(
                        f"WARNING: VALUE_SETS_TO_UPDATE['{target}'] references workspace "
                        f"'{ws_name}' which is not in WORKSPACE_NAMES. Skipping."
                    )

            updated_parts = []
            any_updates = False

            for part in definition["parts"]:
                path = part["path"]
                raw = base64.b64decode(part["payload"]).decode("utf-8")

                target_key = None
                if path == "variables.json" and "default" in VALUE_SETS_TO_UPDATE:
                    target_key = "default"
                elif path.startswith("valueSets/"):
                    # Extract value set name from path (e.g. "valueSets/Env-1D.json" -> "Env-1D")
                    vs_name = path.split("/")[-1].replace(".json", "")
                    if vs_name in VALUE_SETS_TO_UPDATE:
                        target_key = vs_name

                if target_key is not None:
                    target_ws_name = VALUE_SETS_TO_UPDATE[target_key]
                    if target_ws_name in ws_lookup:
                        ws_id, guid_mapping = ws_lookup[target_ws_name]
                        data = json.loads(raw)

                        if path == "variables.json":
                            updates = apply_guid_updates(
                                data["variables"], ws_id, target_ws_name,
                                guid_mapping, f"default (variables.json) <- {target_ws_name}"
                            )
                        else:
                            # Ensure variableOverrides key exists so appends are captured
                            if "variableOverrides" not in data:
                                data["variableOverrides"] = []
                            updates = apply_guid_updates(
                                data["variableOverrides"],
                                ws_id, target_ws_name, guid_mapping,
                                f"{path} <- {target_ws_name}"
                            )

                        if updates:
                            any_updates = True

                        raw = json.dumps(data, indent=2)

                encoded = base64.b64encode(raw.encode("utf-8")).decode("utf-8")
                updated_parts.append(
                    {"path": path, "payload": encoded, "payloadType": "InlineBase64"}
                )

            # POST updated definition back
            if any_updates:
                update_url = (
                    f"{FABRIC_API_BASE}/workspaces/{vl_ws_id}"
                    f"/VariableLibraries/{vl_item_id}/updateDefinition"
                )
                body = {"definition": {"parts": updated_parts}}
                response = poll_lro(
                    requests.post(update_url, headers=HEADERS, json=body), HEADERS
                )

                if response.status_code in (200, 202):
                    print(
                        f"\nVariable Library '{VARIABLE_LIBRARY_NAME}' updated successfully."
                    )
                else:
                    print(
                        f"\nERROR: Update failed: {response.status_code} {response.text}"
                    )
            else:
                print("\nAll targets are already up to date. No update needed.")
else:
    print("\nSkipping value set updates (set UPDATE_VALUE_SETS = True to enable).")